## Rice Image Classification with Deep Convolutional Neural Networks

## Project Topic
This project aims to classify different rice grain varieties using deep convolutional neural networks.  
The selected dataset is the **Rice Image Dataset**, which includes five rice classes: **Arborio, Basmati, Ipsala, Jasmine, and Karacadag**. The dataset contains **75,000 images**, with **15,000 images for each class**. :contentReference[oaicite:0]{index=0}

## Objective
The main objective of this project is to build and compare two image classification models:

1. A custom CNN model built from scratch  
2. A transfer learning model using a pre-trained architecture such as VGG16 or ResNet50

The models will classify rice grain images into their correct variety.

In [35]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization)

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import classification_report, confusion_matrix

In [36]:
import os
import sys
import random
import warnings
from typing import Tuple, List
from pathlib import Path

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchinfo import summary
from torchmetrics.classification import MulticlassConfusionMatrix

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
data_path = Path("/kaggle/input/datasets/muratkokludataset/rice-image-dataset/")
image_path = data_path / "Rice_Image_Dataset" 

ModuleNotFoundError: No module named 'torch'

### Create a dataframe with the Images and Label

In [21]:
 def check_data(dir_path):
    for dirpath, dirnames, filenames in os.walk(dir_path):
        print(f"number of directories: {len(dirnames)} and files: {len(filenames)} image in '{dirpath}'.")

    print("─"*140)
    fig, ax = plt.subplots(1,5, figsize=(15,6))
    for i, file in enumerate(["Arborio", "Basmati", "Ipsala", "Jasmine", "Karacadag"]):
        path = image_path / file
        
        images = list(path.rglob("*.jpg"))
        
        random_image_path = random.choice(images)
        img = Image.open(random_image_path)
        
        ax[i].imshow(img)
        ax[i].set_title(file, fontsize=14)
        ax[i].axis("off")
        plt.tight_layout()
    plt.show()

In [22]:
check_data(image_path)

NameError: name 'image_path' is not defined

In [23]:
def create_dataloader(dataset, train_tf, test_tf, batch_size, num_workers):
    labels = [label for _, label in dataset.samples]
    
    train_idx, test_idx = train_test_split(range(len(dataset)), test_size=0.15, random_state=42, stratify=labels)
    
    train_ds = datasets.ImageFolder(root=image_path, transform=train_tf)
    test_ds = datasets.ImageFolder(root=image_path, transform=test_tf)

    class_names = train_ds.classes
    train_ds = Subset(train_ds, train_idx)
    test_ds = Subset(test_ds, test_idx)

    train_dataloader = DataLoader(train_ds, shuffle=True, batch_size=batch_size, num_workers=num_workers, pin_memory=True)
    test_dataloader = DataLoader(test_ds, shuffle=False, batch_size=batch_size, num_workers=num_workers, pin_memory=True)

    return train_dataloader, test_dataloader, class_names, test_ds


In [24]:
dataset = datasets.ImageFolder(root=image_path)
BATCH_SIZE = 16
NUM_WORKERS = os.cpu_count()

train_tf = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(p=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_tf = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train_dataloader, test_dataloader, class_names, test_ds = create_dataloader(dataset, train_tf, test_tf, BATCH_SIZE, NUM_WORKERS)


NameError: name 'datasets' is not defined

In [25]:
class RiceCNN(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()

        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),                       # After Conv1:  torch.Size([32, 128, 128, 128])
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),                       # After Conv2:  torch.Size([32, 128, 128, 128])
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)       # After pool1:  torch.Size([32, 128, 64, 64])
        )

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3, stride=1,
                      padding=1),                       # After Conv3:  torch.Size([32, 128, 64, 64])
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),                       # After Conv4:  torch.Size([32, 128, 64, 64])
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)       # After pool2:  torch.Size([32, 128, 32, 32])
        )

        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(in_features=hidden_units * 32 * 32, out_features=output_shape))

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.conv_block_2(self.conv_block_1(X)))


NameError: name 'nn' is not defined

In [26]:
def train_step(
    model: nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device
):
    model.train()

    train_loss, train_acc = 0, 0
    
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        y_pred = model(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_pred_class = torch.argmax(y_pred, dim=1)
        train_acc += (y_pred_class == y).sum().item()

    train_loss /= len(dataloader)
    train_acc /= len(dataloader.dataset)
        
    return train_loss, train_acc

def test_step(
    model: nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: nn.Module,
    device: torch.device
):
    model.eval()

    test_loss, test_acc = 0, 0

    all_preds = []
    all_targets = []

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
    
            test_pred = model(X)
    
            loss = loss_fn(test_pred, y)
            test_loss += loss.item()

            test_pred_labels = test_pred.argmax(dim=1)
            test_acc += (test_pred_labels == y).sum().item()

            all_preds.extend(test_pred_labels.cpu().numpy())
            all_targets.extend(y.cpu().numpy())
            
    test_loss /= len(dataloader)
    test_acc /= len(dataloader.dataset)

    return test_loss, test_acc, all_preds, all_targets

def train(
    model: nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: torch.device,
    scheduler: torch.optim.lr_scheduler,
    epochs: int=5
):
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": [],
        "all_preds": [],
        "all_targets": []
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)

        test_loss, test_acc, all_preds, all_targets = test_step(model=model,
                                                               dataloader=test_dataloader,
                                                               loss_fn=loss_fn,
                                                               device=device)
        if scheduler is not None:
            try:
                scheduler.step(test_loss)
            except TypeError:
                scheduler.step()

        print(
            f"Epoch: {epoch + 1}/{epochs} | " 
            f"Train Loss: {train_loss:.4f} | "
            f"Train Accuracy: {train_acc:.4f} | " 
            f"Test Loss: {test_loss:.4f} | "
            f"Test Accuracy: {test_acc:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']}"
            )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)
        results["all_preds"].append(all_preds)
        results["all_targets"].append(all_targets)

    return results

NameError: name 'nn' is not defined

In [27]:
model = RiceCNN(input_shape=3, hidden_units=128, output_shape=len(class_names)).to(device)

summary(model, input_size=[32, 3, 128, 128], col_names=["input_size", "output_size", "trainable"])

NameError: name 'RiceCNN' is not defined

In [28]:
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.02)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2)

results = train(model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                device=device,
                scheduler=scheduler,
                epochs=8)

NameError: name 'nn' is not defined

In [30]:
def plot_loss_curve(results, title=""):
    train_loss = results["train_loss"]
    test_loss = results["test_loss"]

    train_accuracy = results["train_acc"]
    test_accuracy = results["test_acc"]

    epochs = range(len(results["train_loss"]))

    plt.figure(figsize=(15,5))
    sns.set_theme(style=("whitegrid"))
    plt.suptitle(title, fontsize=18, fontweight="bold")
    
    plt.subplot(1,2,1)
    plt.plot(epochs, train_loss, label="Train Loss")
    plt.plot(epochs, test_loss, label="Test Loss")   
    plt.title("Loss")
    plt.xlabel("Epochs")
    plt.legend()
    
    plt.subplot(1,2,2)
    plt.plot(epochs, train_accuracy, label="Train Accuracy")
    plt.plot(epochs, test_accuracy, label="Test Accuracy")
    plt.title("Accuracy")
    plt.xlabel("Epochs")
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [31]:
def confusion_matrix(results):
    fig, ax = plt.subplots(1,1, figsize=(10,4))
    confusion_matrix = MulticlassConfusionMatrix(num_classes=5)
    all_preds = torch.tensor(results["all_preds"])
    all_targets = torch.tensor(results["all_targets"])
    matrix = confusion_matrix(all_targets, all_preds)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_xticks(ticks=np.arange(len(class_names)), labels=class_names, fontweight="bold", ha="center", rotation=20)
    ax.set_yticks(ticks=np.arange(len(class_names)), labels=class_names, fontweight="bold", rotation=-20)
    plt.show()

In [33]:
def show_random_predictions(model, dataset, class_names, n=9, device=device):
    model.eval()
    
    plt.figure(figsize=(15, 6))

    indices = random.sample(range(len(dataset)), n)

    with torch.inference_mode():
        for i, idx in enumerate(indices):
            img, true_label = dataset[idx]

            img_input = img.unsqueeze(0).to(device)  
            logits = model(img_input)
            pred_label = logits.argmax(dim=1).item()

            old_stderr = sys.stderr
            sys.stderr = open(os.devnull, "w")

            img_show = img.permute(1, 2, 0)
                        
            correct = (pred_label == true_label)
            color = "green" if correct else "red"

            plt.subplot(3, 3, i + 1)
            plt.imshow(img_show)
            plt.axis("off")

            plt.title(
                f"Pred: {class_names[pred_label]}\nTrue: {class_names[true_label]}",
                color=color,
                fontsize=12,
                fontweight="bold"
            )

            sys.stderr = old_stderr
            
    plt.tight_layout()
    plt.show()



NameError: name 'device' is not defined

In [34]:
plot_loss_curve(results)
confusion_matrix(results)
show_random_predictions(model, test_ds, class_names, n=9, device=device)

NameError: name 'results' is not defined